# Bike Demand Prediction — Experimentation Notebook

Scratch notebook for feature engineering, modeling, and MLflow-tracked experiments.
Once the approach is settled here, logic gets modularized into `src/`.

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)


## 1. Load raw data

In [ ]:
df = pd.read_csv('../data/raw/hour.csv', parse_dates=['dteday'])
df = df.sort_values(['dteday', 'hr']).reset_index(drop=True)
df.head()


In [ ]:
df.info()
df.describe()


## 2. Chronological train/test split

No random shuffling — this is time-ordered data. Reserve a contiguous test window at the end of the timeline for the replay/monitoring dashboard.

In [ ]:
# TODO: decide and document the split boundary (e.g. last N weeks reserved as test window)
split_date = None  # e.g. pd.Timestamp('2012-11-01')

# train_df = df[df['dteday'] < split_date]
# test_df = df[df['dteday'] >= split_date]


## 3. Feature engineering

Build all features here first; once finalized, move into a shared `src/features.py` module used by both training and the replay script (no duplicated logic).

- Calendar/time features (hour, day-of-week, month, season, holiday, workingday, cyclical sin/cos)
- Weather features (use dataset's own recorded/normalized columns)
- Look-left aggregates (trailing avg same hour+weekday, same-hour fallback, prior-year same hour/season, EWMA trend)
- Cold-start handling for earliest rows
- Leakage check: confirm every feature only uses data strictly before its own timestamp

In [ ]:
# Calendar / cyclical features
def add_calendar_features(data):
    data = data.copy()
    data['hour_sin'] = np.sin(2 * np.pi * data['hr'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hr'] / 24)
    data['dow_sin'] = np.sin(2 * np.pi * data['weekday'] / 7)
    data['dow_cos'] = np.cos(2 * np.pi * data['weekday'] / 7)
    return data

df = add_calendar_features(df)
df.head()


In [ ]:
# TODO: look-left / trailing aggregate features (leakage-safe: use .shift() before any rolling/expanding window)


## 4. Modeling

Train on the training slice only. Freeze the chosen model — it does not get retrained during later replay/monitoring steps.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import lightgbm as lgb

# TODO: define feature_cols and target once features are finalized
# X_train, y_train = train_df[feature_cols], train_df['cnt']
# X_test, y_test = test_df[feature_cols], test_df['cnt']


## 5. MLflow experiment tracking

Log params (feature set, window lengths, hyperparameters) and metrics (MAE, RMSE, MAPE) for every run.

In [ ]:
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('bike-demand-prediction')

with mlflow.start_run(run_name='baseline'):
    # TODO: fit model, log params/metrics, log model
    # mlflow.log_params({...})
    # mlflow.log_metrics({'mae': mae, 'rmse': rmse, 'mape': mape})
    # mlflow.lightgbm.log_model(model, 'model')
    pass


## 6. Notes / decisions log

Track key decisions made during experimentation (window lengths chosen, features dropped, etc.) so they can be carried into the modularized code and README.